In [ ]:
code = 'LONG_BUTTERFLY_SPREAD'
pickle_path = 'P:/PGC Data/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output/'
day_data_dir = "C:/Users/pgc_0/OneDrive/Desktop/pgcbacktest/INTRADAY CODES/Long Butterfly Spread/index_day_data"

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def prev_ohlc(spot_day_data, date):
    prev = spot_day_data[spot_day_data["datetime"] < pd.to_datetime(date)]
    return prev.loc[prev["datetime"].idxmax(), ["datetime","open","high","low","close"]]

def pivot_levels(H, L, C):
    P = (H + L + C) / 3

    R1 = (2 * P) - L
    R2 = P + (H - L)
    R3 = H + 2 * (P - L)
    R4 = R3 + (R2 - R1)
    R5 = R4 + (R3 - R2)

    S1 = (2 * P) - H
    S2 = P - (H - L)
    S3 = L - 2 * (H - P)
    S4 = S3 - (S1 - S2)
    S5 = S4 - (S2 - S3)

    # round only here, after all math is done
    pivot_prices = [round(x, 2) for x in [S5, S4, S3, S2, S1, P, R1, R2, R3, R4, R5]]
    return pivot_prices

In [ ]:
def BUTTERFLY(bt, start_time, end_time, std_premium_prct, spot_day_data):
    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)
        prev_date, O, H, L, C = prev_ohlc(spot_day_data, bt.current_date)
        
        pivot_prices = pivot_levels(H, L, C)
        S5, S4, S3, S2, S1, P, R1, R2, R3, R4, R5 = pivot_prices
        
        std_ce_scrip, std_pe_scrip, std_ce_price, std_pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om='ATMS')
        if std_ce_scrip is None: return None
        
        entry_time = start_dt
        check_data = bt.spot_data.copy().reset_index()
        check_data = check_data[(check_data['date_time'] > start_dt) & (check_data['date_time'] < end_dt)]
        
        closes = check_data['close'].values
        datetimes = check_data['date_time'].values
        
        std_strike = get_strike(std_ce_scrip)
        std_price = std_ce_price + std_pe_price
        
        wing_width = (std_price * std_premium_prct)/100
        wing_width = max(round(wing_width/bt.gap)*bt.gap, bt.gap)
        
        current_close = bt.spot_data.loc[start_dt, 'close']
        
        if current_close > R5:
            print('Current Price is higher than R5 => No Trade !!!')
            return
        
        if current_close < S5:
            print('Current Price is lower than S5 => No Trade !!!')
            return
            
        if current_close < P:
            signal = "PE"
            
            if wing_width == bt.gap:
                body_strike = get_strike(std_ce_scrip) - wing_width
            else:
                body_strike = (get_strike(std_ce_scrip) + bt.gap) - wing_width
            
            lower_wing = f"{int(body_strike + wing_width)}PE"
            body_wing = f"{int(body_strike)}PE"
            upper_wing = f"{int(body_strike - wing_width)}PE"
            
            target = body_strike
            stoploss = [p for p in pivot_prices if p > current_close][1]
            
            mask_target = closes < target
            mask_stoploss = closes > stoploss

        elif current_close > P:
            signal = "CE"
            
            body_strike = get_strike(std_ce_scrip) + wing_width
            
            lower_wing = f"{int(body_strike - wing_width)}CE"
            body_wing = f"{int(body_strike)}CE"
            upper_wing = f"{int(body_strike + wing_width)}CE"
            
            target = body_strike
            stoploss = [p for p in pivot_prices if p < current_close][-2]
            
            mask_target = closes > target
            mask_stoploss = closes < stoploss
            
        else:
            return
        
        combined_mask = mask_target | mask_stoploss

        if combined_mask.any():
            idx = np.argmax(combined_mask)
            exit_time = pd.Timestamp(datetimes[idx])
        else:
            exit_time = end_dt
        
        _, _, _, _, _, lower_wing_pnl = bt.sl_check_single_leg(start_dt, exit_time, lower_wing, orderside="BUY")
        _, _, _, _, _, body_wing_pnl = bt.sl_check_single_leg(start_dt, exit_time, body_wing, orderside="SELL")
        _, _, _, _, _, upper_wing_pnl = bt.sl_check_single_leg(start_dt, exit_time, upper_wing, orderside="BUY")    
        
        total_pnl = lower_wing_pnl + (2 * body_wing_pnl) + upper_wing_pnl
        
        return [code, bt.index, start_time, end_time, std_premium_prct, bt.current_date.date(), bt.current_date.day_name(), bt.dte, entry_time.time(), future_price, current_close, std_strike, std_price, H, L, C, S5, S4, S3, S2, S1, P, R1, R2, R3, R4, R5, signal, wing_width, lower_wing, body_wing, upper_wing, target, stoploss, lower_wing_pnl, body_wing_pnl, upper_wing_pnl, total_pnl] 
    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, std_premium_prct])
        return

In [ ]:
for row_idx in range(len(meta_data)):

    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, dte, from_date, to_date, start_time, end_time, date_lists = get_meta_row_data(meta_row, pickle_path)

            log_cols = ('P_Strategy/P_Index/P_StartTime/P_EndTime/P_StdPremiumPrct/Date/Day/DTE/EntryTime/Future/Current.Close/STD.Strike/STD.Price/Prev.High/Prev.Low/Prev.Close/S5/S4/S3/S2/S1/P/R1/R2/R3/R4/R5/Signal/Wing.Width/Lower.Wing/Body.Wing/Upper.Wing/Target/Stoploss/Lower.Wing.PNL/Body.Wing.PNL/Upper.Wing.PNL/Total.PNL').split('/')

            for current_date in date_lists:

                file_name = f"{index} {current_date.date()} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    bt = IntradayBacktest(pickle_path, index, current_date, dte, start_time, end_time)
                    spot_day_data = pd.read_parquet(f"{day_data_dir}/{index}.parquet")
                    
                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)

                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [BUTTERFLY(bt, row.entry_time, row.exit_time, row.std_premium_prct, spot_day_data) for row in tqdm(chunk_parameter.itertuples(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)
                    
                    t2 = datetime.datetime.now()
                    print(t2-t1)
        except Exception as e:
            input(str(e))